In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix



In [ ]:
df = sns.load_dataset('titanic')
print(df.head())

In [ ]:
data = df.drop(['deck','alive','who','adult_male','class','embark_town'],axis=1)

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data['survived'].value_counts(normalize=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

to_categorical_columns = ['sibsp','pclass','parch']
data[to_categorical_columns] = data[to_categorical_columns].astype('category')

numeric_columns = data.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

# pandas 최신 버전 대응
for col in data.select_dtypes(include=['object', 'string']).columns:
    data[col] = data[col].astype('category')

categorical_columns = data.select_dtypes(
    include=['object', 'category', 'bool']
).columns.tolist()

numeric_columns.remove('survived')

print("Numeric Columns 단변량 분석\n")
for col in numeric_columns:
    plt.figure(figsize=(12, 4))

    # Histogram
    plt.subplot(1, 2, 1)
    sns.histplot(
        data[col],
        kde=True
    )
    plt.title(f'{col} Histogram')

    # Boxplot
    plt.subplot(1, 2, 2)
    sns.boxplot(
        x=data[col]
    )
    plt.title(f'{col} Boxplot')

    plt.tight_layout()
    plt.show()

print("="*50)
print("categorical columns 단변량 분석\n")
for col in categorical_columns:
    plt.figure(figsize=(8, 4))

    sns.countplot(
        x=data[col],
        order=data[col].value_counts().index
    )

    plt.title(f'{col} Countplot')
    plt.xticks(rotation=45)

    plt.tight_layout()
    plt.show()

흠 일단 자료들의 분포를 보니 fare hist는 오른쪽으로 꼬리가 아주 긴 분포네, log변환 해보고싶긴하다.
 그리고 categori들 pclass, sibsp, parch는 각 자료간에 위계가 존재하기때문에 ordinal encoding 고려해볼만하고,
 그외는 onehotencoding 해주면 될듯

# Titanic Dataset 컬럼 설명 (현재 사용 기준)

## Target
- **survived**
  - 생존 여부
  - `0 = 사망`, `1 = 생존`

---

## Numeric Features

- **age**
  - 승객 나이
  - 일부 결측치 존재
  - 연속형 변수

- **fare**
  - 티켓 요금
  - 오른쪽 꼬리가 긴 분포 (log 변환 고려 가능)

- **sibsp**
  - 함께 탑승한 형제자매 / 배우자 수
  - 가족 규모 관련 feature

- **parch**
  - 함께 탑승한 부모 / 자녀 수
  - 가족 규모 관련 feature

---

## Categorical Features

- **pclass**
  - 객실 등급
  - `1 = 1등석`, `2 = 2등석`, `3 = 3등석`
  - 사회경제적 수준 반영 가능
  - 순서형 특성 존재

- **sex**
  - 성별
  - `male / female`

- **embarked**
  - 탑승 항구
  - `C = Cherbourg`
  - `Q = Queenstown`
  - `S = Southampton`

---

## 제거한 컬럼들

- **deck**
  - 객실 층 정보
  - 결측치 과다

- **alive**
  - survived와 중복

- **class**
  - pclass와 중복

- **who**
  - sex + age 기반 파생 변수

- **adult_male**
  - sex + age 중복성 높음

- **embark_town**
  - embarked와 중복

---

## Feature Engineering 후보

- **family_size**
```python
sibsp + parch + 1

In [ ]:
data = df.drop(['deck','alive','who','adult_male','class','embark_town'],axis=1)
data['alone'] = data['alone'].astype(int)

In [ ]:
#파생변수 설정
# data['family_size'] = data['sibsp'] + data['parch'] + 1

In [ ]:
X = data.drop('survived', axis=1)
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
# feature selection
numeric_features = ['age', 'fare', 'sibsp', 'parch','alone']
categorical_features = ['pclass', 'sex', 'embarked']

In [ ]:
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

baseline_pipe = Pipeline([
    ('preprocessor', preprocessor),
    (
        'model', LogisticRegression(max_iter=1000,
                                    random_state=42)
    )
])

cv_scores = cross_validate(
    baseline_pipe,
    X_train,
    y_train,
    cv=5,
    scoring=['accuracy','f1','precision','recall','roc_auc']
)
print("Accuracy : ",cv_scores['test_accuracy'].mean())
print("F1 Score : ",cv_scores['test_f1'].mean())
print("Precision : ",cv_scores['test_precision'].mean())
print("Recall : ",cv_scores['test_recall'].mean())
print("ROC AUC : ",cv_scores['test_roc_auc'].mean())



In [ ]:
baseline_pipe.fit(X_train,y_train)
y_pred = baseline_pipe.predict(X_test)

print("test_Acc:", baseline_pipe.score(X_test,y_test))
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
def evaluate_pipe (pipe, X_train, y_train, X_test, y_test):
    cv_scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=5,
        scoring=['accuracy','f1','precision','recall','roc_auc']
    )
    print("Accuracy : ",cv_scores['test_accuracy'].mean())
    print("F1 Score : ",cv_scores['test_f1'].mean())
    print("Precision : ",cv_scores['test_precision'].mean())
    print("Recall : ",cv_scores['test_recall'].mean())
    print("ROC AUC : ",cv_scores['test_roc_auc'].mean())

    pipe.fit(X_train,y_train)
    y_pred = pipe.predict(X_test)
    print("test_Acc:", pipe.score(X_test,y_test))
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

In [ ]:
# fare feature log변환 해보자
numeric_features = ['age', 'sibsp', 'parch','alone']
fare_feature = ['fare']
categorical_features = ['pclass', 'sex', 'embarked']

In [ ]:
fare_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('log', FunctionTransformer(np.log1p, validate=False)),
    ('scaler', StandardScaler())
])

log_preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features),
    ('fare', fare_transformer, fare_feature)
])

log_pipe = Pipeline([
    ('preprocessor', log_preprocessor),
    ('model', LogisticRegression(max_iter=1000,))
])

evaluate_pipe(log_pipe, X_train, y_train, X_test, y_test)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from my_ml_kit import (
    run_model_candidates,
    compare_results,
    print_test_report
)

In [ ]:
X = data.drop('survived', axis=1)
y = data['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
numeric_features = ['age', 'fare', 'sibsp', 'parch', 'alone']
categorical_features = ['pclass', 'sex', 'embarked']

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [ ]:
basic_models = {
    'logistic': LogisticRegression(
        max_iter=1000,
        random_state=42
    ),

    'decision_tree': DecisionTreeClassifier(
        random_state=42
    ),

    'knn': KNeighborsClassifier(),

    'gaussian_nb': GaussianNB()
}

In [ ]:
results = []

results, trained_pipes = run_model_candidates(
    models=basic_models,
    preprocessor=preprocessor,
    X_train=X_train,
    y_train=y_train,
    X_test=X_test,
    y_test=y_test,
    results=results,
    cv=5
)

In [ ]:
results_df = compare_results(
    results,
    sort_by='cv_roc_auc'
)

results_df

In [ ]:
for model_name in basic_models.keys():
    print(f"\n===== {model_name} =====")

    print_test_report(
        trained_pipes[model_name],
        X_test,
        y_test
    )

In [ ]:
baseline_pipe.get_params()

In [ ]:
trained_pipes['logistic'].get_params()